In [1]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from collections import deque
import numpy as np
from torch import optim, nn
import torch 
import matplotlib.pyplot as plt
import os, sys, random, time
from myQnet import QNetwork

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
device = torch.device("cpu")
print(f"device: {device}")

main_dir = os.getcwd()
video_dir = os.path.join(main_dir, "videos/cartpole")
model_dir = os.path.join(main_dir, "models/cartpole")

if not os.path.exists(video_dir):
    os.makedirs(video_dir)
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

render_env = gym.make("CartPole-v1")  

device: cpu


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int32),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32)
        )
    def __len__(self):
        return len(self.buffer)

class CartPoleAgent:
    def __init__(
        self,  
        device,
        video_dir,
        model_dir,
        hidden_dim=32, 
        num_episodes=1000,
        lr = 5e-4,
        gamma=0.98, 
        epsilon=1.0, 
        epsilon_min=0.01, 
        epsilon_decay=0.99, 
        batch_size=64, 
        eval_freq=50,  
        patience=20, 
        min_improvement=5.0):

        self.env = gym.make("CartPole-v1")
        self.device = device
        self.video_dir = video_dir
        self.model_dir = model_dir
        self.hidden_dim = hidden_dim
        self.num_episodes = num_episodes
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size 
        self.patience = patience
        self.min_improvement = min_improvement
        self.improvement_count = 0
        self.best_eval = -float('inf')
        self.eval_freq = eval_freq
        self.lr = lr 

        self.state_dim = self.env.observation_space.shape[0]
        self.action_dim = self.env.action_space.n

        self.qnet = QNetwork(self.state_dim, self.action_dim, hidden_dim=hidden_dim).to(device)
        self.target_net = QNetwork(self.state_dim, self.action_dim, hidden_dim=hidden_dim).to(device)
        self.target_net.load_state_dict(self.qnet.state_dict())
        self.optimizer = optim.Adam(self.qnet.parameters(), lr=self.lr)

        self.buffer = ReplayBuffer(capacity=10000) 

        self.episode_rewards = []
        self.episode_losses = []
        self.episode_epsilons = []

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            q_values = self.qnet(state)
            return q_values.argmax(dim=1).item()

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return None
        state, action, reward, next_state, done = self.buffer.sample(self.batch_size)
        states = torch.tensor(state, dtype=torch.float32, device=self.device)
        actions = torch.tensor(action, dtype=torch.int32, device=self.device).unsqueeze(1)
        rewards = torch.tensor(reward, dtype=torch.float32, device=self.device).unsqueeze(1)
        next_states = torch.tensor(next_state, dtype=torch.float32, device=self.device)
        dones = torch.tensor(done, dtype=torch.float32, device=self.device).unsqueeze(1)

        q_values = self.qnet(states).gather(1, actions)
        with torch.no_grad():
            next_q = self.target_net(next_states).max(dim=1, keepdim=True)[0]
            target = rewards + self.gamma * (1 - dones) * next_q
        loss = nn.MSELoss()(q_values, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def evaluate(self):
        rewards = []
        for _ in range(self.eval_freq):
            state, _ = self.env.reset()
            done = False
            total_reward = 0
            while not done:
                state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
                with torch.no_grad():
                    action = self.qnet(state).argmax(dim=1).item()
                next_state, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated
                state = next_state
                total_reward += reward
            rewards.append(total_reward)
        return np.mean(rewards), np.std(rewards)

    def record(self, eps=3, name_prefix=f"cartpole_"):
        env = gym.make("CartPole-v1", render_mode="rgb_array")
        env = RecordVideo(
            env,
            self.video_dir,
            episode_trigger=lambda ep: True,
            name_prefix=name_prefix
        )
        self.qnet.eval()
        for ep in range(eps):
            state, _ = env.reset()
            done = False
            while not done:
                with torch.no_grad():
                    action = self.qnet(torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)).argmax(1).item()
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                state = next_state
            env.close()
            self.qnet.train()

    def train(self): 
        for epi in range(self.num_episodes):
            state, _ = self.env.reset()
            done = False
            total_reward = 0
            losses = []

            while not done:
                action = self.select_action(state)
                next_state, reward, terminated, truncated, info = self.env.step(action)
                done = terminated or truncated
                
                self.buffer.push(state, action, reward, next_state, done)

                loss = self.train_step()

                if loss is not None:
                    losses.append(loss)

                state = next_state
                total_reward += reward

            self.epsilon=max(self.epsilon_min, self.epsilon*self.epsilon_decay)
            avg_loss = np.mean(losses) if losses else 0

            mean_eval, std_eval = self.evaluate()
            self.episode_rewards.append(mean_eval)
            self.episode_losses.append(avg_loss)
            self.episode_epsilons.append(self.epsilon)

            if epi % self.eval_freq == 0:
                mean_eval, std_eval = self.evaluate()
                if mean_eval > self.best_eval:
                    self.best_eval = mean_eval
                    self.target_net.load_state_dict(self.qnet.state_dict()) 
                    self.improvement_count = 0
                    torch.save(
                        self.qnet.state_dict(), 
                        os.path.join(self.model_dir, 
                        f"best_cartpole_{self.hidden_dim}.pth"))
                    print(f"Episode {epi:3d}: Mean reward {mean_eval:8.2f}, Std reward {std_eval:8.2f}, avg loss {avg_loss:8.4f}, epsilon {self.epsilon:.3}")
                    self.record(eps=3, name_prefix=f"cartpole_{self.hidden_dim}_at_{epi}")
                else: 
                    print(f"No improvement on episode {epi}")
                if self.improvement_count >= self.patience:
                    print(f"Early stopping on episode {epi}, no improvement in {self.patience} episodes")
                    break

    def show_data(self):

        def ma(data, window=10, mode='valid'):
            return np.convolve(np.array(data).flatten(), np.ones(window) / window, mode=mode)

        plt.plot(ma(self.episode_rewards), c='black', lw=1, label= f'MA')
        plt.plot(self.episode_rewards, c='gray', alpha=0.5, lw=0.5, label='Reward')
        plt.xlabel('Episode')
        plt.ylabel('Reward')
        plt.legend()
        plt.savefig(os.path.join(self.model_dir, f'reward_{self.hidden_dim}.png'))
        np.save(os.path.join(self.model_dir, f'reward_{self.hidden_dim}.npy'), self.episode_rewards)

        plt.plot(ma(self.episode_losses), c='black', lw=1, label='MA')
        plt.plot(self.episode_losses, c='gray', alpha=0.5, lw=0.5, label='Loss')
        plt.xlabel('Episode')
        plt.ylabel('Loss')
        plt.legend()
        plt.savefig(os.path.join(self.model_dir, f'loss_{self.hidden_dim}.png'))
        np.save(os.path.join(self.model_dir, f'loss_{self.hidden_dim}.npy'), self.episode_losses)

        plt.plot(self.episode_epsilons, c='black', lw=1, label='Epsilon') 
        plt.xlabel('Episode')
        plt.ylabel('Epsilon')
        plt.legend()
        plt.savefig(os.path.join(self.model_dir, f'epsilon_{self.hidden_dim}.png'))
        np.save(os.path.join(self.model_dir, f'epsilon_{self.hidden_dim}.npy'), self.episode_epsilons)

In [ ]:
if __name__ == "__main__":

    hidden_dim = 512
    num_episodes = 10000
    agent = CartPoleAgent(
        device, 
        video_dir,
        model_dir, 
        lr = 1e-3,
        gamma=0.98, 
        epsilon=1.0, 
        epsilon_min=0.01, 
        epsilon_decay=0.995, 
        batch_size=64, 
        eval_freq=50,  
        patience=30, 
        min_improvement=1.0,
        hidden_dim=hidden_dim, num_episodes=10000)
    agent.train()
 

Episode   0: Mean reward     9.46, Std reward     0.81, avg loss   0.0000, epsilon 0.995


/opt/miniconda3/envs/pytorch_csiro/lib/python3.10/site-packages/gymnasium/wrappers/rendering.py:296: UserWarning: WARN: Overwriting existing videos at /Users/kaheichoi/git_repo/reinforce/videos/cartpole folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


No improvement on episode 50
Episode 100: Mean reward     9.50, Std reward     0.85, avg loss   0.0003, epsilon 0.603
Episode 150: Mean reward    10.96, Std reward     1.22, avg loss   0.0159, epsilon 0.469
Episode 200: Mean reward    23.40, Std reward     1.78, avg loss   0.0313, epsilon 0.365
Episode 250: Mean reward   100.84, Std reward    78.06, avg loss   0.0278, epsilon 0.284
No improvement on episode 300
No improvement on episode 350
Episode 400: Mean reward   118.56, Std reward    15.83, avg loss   0.0207, epsilon 0.134
No improvement on episode 450
No improvement on episode 500
No improvement on episode 550
No improvement on episode 600
No improvement on episode 650
No improvement on episode 700
No improvement on episode 750
No improvement on episode 800
No improvement on episode 850
No improvement on episode 900
No improvement on episode 950
Episode 1000: Mean reward   120.42, Std reward     5.78, avg loss   0.0090, epsilon 0.01
Episode 1050: Mean reward   160.12, Std reward 

FileNotFoundError: [Errno 2] No such file or directory: '/Users/kaheichoi/git_repo/reinforce/models/cartpole/reward_512.npy'

In [ ]:
np.save(os.path.join(model_dir, f'reward_{hidden_dim}.npy'), episode_rewards)
np.save(os.path.join(model_dir, f'loss_{hidden_dim}.npy'), episode_losses)
np.save(os.path.join(model_dir, f'epsilon_{hidden_dim}.npy'), episode_epsilons)
agent.show_data(episode_rewards, episode_losses, episode_epsilons)

NameError: name 'episode_rewards' is not defined

In [11]:
agent.show_data()

TypeError: CartPoleAgent.show_data() missing 3 required positional arguments: 'episode_rewards', 'episode_losses', and 'episode_epsilons'